In [2]:
%pip install mlflow

Defaulting to user installation because normal site-packages is not writeable

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

  Using cached mlflow-3.18.0-py3-none-any.whl.metadata (50 kB)

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

  Using cached mlflow_skinny-3.18.0-py3-none-any.whl.metadata (51 kB)

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

... (13 outputs left out)

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

  Using cached matplotlib-3.11.2-cp311-cp311-win_amd64.whl.metadata (80 kB)


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

  Using cached pyarrow-26.0.1-cp311-cp311-win_amd64.whl.metadata (3.0 kB)
  Using cached skops-0.16.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached waitress-3.0.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached cachetools-7.3.2-py3-none-any.whl.metadata (5.5 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached databricks_sdk-0.167.0-py3-none-any.whl.metadata (44 kB)
  Using cached fastapi-0.163.1-py3-none-any.whl.metadata (27 kB)
  Using cached importlib_metadata-10.2.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached opentelemetry_api-1.45.2-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_proto-1.45.2-py3-none-any.whl.metadata (2.3 kB)
  Using cached opentelemetry_sdk-1.45.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached python_multipart-0.0.33-py3-none-any.whl.metadata (6.3 kB)
  Using cached sqlparse-0.6.1-py3-none-any.whl.metadata (6.9 kB)
  Using cached starlette-1.10.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached uv

In [3]:
import pandas as pd
import numpy as np
import sklearn
import dvc
import mlflow

print("pandas : " , pd.__version__)
print("Numpy : " , np.__version__)
print("Sklearn : " , sklearn.__version__)
print("mlflow : " , mlflow.__version__)

C:\Users\opc\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


pandas :  3.1.1
Numpy :  2.4.2
Sklearn :  1.8.0
mlflow :  3.18.0


In [4]:
import os
print(os.getcwd())

C:\Users\opc\customer-churn-mlops-project


In [4]:
os.chdir(r"C:\Users\opc\customer-churn-mlops-project")
print(os.getcwd())

C:\Users\opc\customer-churn-mlops-project


In [5]:
from sklearn.model_selection import train_test_split  # 7043 rows - training set, testing set
from sklearn.preprocessing import StandardScaler, OneHotEncoder # dataset - cols - higher values (area)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, confusion_matrix, classification_report

In [6]:
df = pd.read_csv(r"C:\Users\opc\Downloads\WA_Fn-UseC_-Telco-Customer-Churn.csv")
print("Dataset loaded successfully")
print("Dataset shape : ", df.shape)
df.head() # prints the first 5 rows of the datasetprint("Dataset loaded successfully")
print("Dataset shape : ", df.shape)
df.head() # prints the first 5 rows of the dataset

Dataset loaded successfully
Dataset shape :  (7043, 21)
Dataset shape :  (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [7]:
# Churn Percentage  : churn col - yes no
print(df["Churn"].value_counts())
print("Churn Percentage")
print(df["Churn"].value_counts(normalize=True) * 100)

Churn
No     5174
Yes    1869
Name: count, dtype: int64
Churn Percentage
Churn
No     73.463013
Yes    26.536987
Name: proportion, dtype: float64


In [8]:
# Sepearate features and target
X = df.drop("Churn", axis=1)
y = df["Churn"].map({
                      "No" : 0,
                      "Yes" : 1
})
print("Input Features Shape : ", X.shape)
print("Target Shape : ", y.shape)

Input Features Shape :  (7043, 20)
Target Shape :  (7043,)


In [9]:
# Identify numerical and categorical features
numerical_features = X.select_dtypes(
            include=["int64","float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
            include=["object"]
).columns.tolist()

print(numerical_features)
print("---------------------")
print(categorical_features)

C:\Users\opc\AppData\Local\Temp\2\ipykernel_5800\2041482257.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


['SeniorCitizen', 'tenure', 'MonthlyCharges']
---------------------
['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges']


In [10]:
# Split the dataset into training and testing
# train_test_split
# X and y -> X_train, X_test, y_train, y_test
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state = 42, stratify = y)
                     # testset - 20%, trainset - 80%
                     # random_state - same split 
                    # stratify = y, maintain approx same churn/non churn proportion
print("Training Set : ", X_train.shape[0])
print("Testing Set : ", X_test.shape[0])

Training Set :  5634
Testing Set :  1409


In [11]:
# Transformations
# numerical - StandardScaler - higher and lower values should be treated equally
# OneHotEncoder
# ColumnTransformer - apply correct preprocessing to each group

from sklearn.compose import ColumnTransformer
preprocessor = ColumnTransformer(
       transformers = [
           ("num",
             StandardScaler(),
             numerical_features
           ),
           ("cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
           )
       ]
)
print("Preprocessing Pipeline created")

Preprocessing Pipeline created


In [12]:
# Train the LogisticRegression Model
logistic_model = Pipeline(
    steps = [
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(max_iter = 1000, class_weight = "balanced")
        )
    ]
)

# Train the Model
logistic_model.fit(X_train, y_train)
print("logistic_model trained")

logistic_model trained


In [13]:
# Evaluate the Model
y_pred = logistic_model.predict(X_test) # predictions done by model
# Actual Values - y_test
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
print("Accuracy : ", accuracy)
print("Precision : ", precision)

Accuracy :  0.7601135557132718
Precision :  0.5358565737051793


In [16]:
rf_model = Pipeline(
    steps = [
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, class_weight="balanced")
                                 # no. of decision tress
        )
    ]
)

# Train the Model
rf_model.fit(X_train, y_train)
print("rf model trained")

rf model trained


In [17]:
rf_pred = rf_model.predict(X_test)
accuracy_rf = accuracy_score(y_test, rf_pred)
precision_rf = precision_score(y_test, rf_pred)
print("Accuracy RF: ", accuracy_rf)
print("Precision RF : ", precision_rf)

Accuracy RF:  0.7040454222853088
Precision RF :  0.46635367762128327
